# 19. Data Collection and Crawling

## Problem

训练数据不是凭空出现的。网页、代码仓库、论文、论坛和文档这些原始内容，是怎样从互联网和数据源里被抓出来、抽出来、存下来，再变成后续清洗管线的原材料？

## Dependencies

建议先读 pretraining 和多阶段训练那几节。因为只有把数据放回完整训练链里，才会知道“采集策略”为什么本身就是模型质量的一部分。


## Why data collection is not just downloading text

很多人会把数据采集想成：

- 找一批网址
- 把 HTML 下载下来
- 抽出文字
- 结束

这太低估真实工程了。

真正的数据采集系统要同时处理很多事：

- 来源管理：哪些源值得抓，优先级怎么排
- 抓取策略：多久抓一次，失败怎么重试，频率怎么控
- 页面抽取：正文和导航、广告、脚本怎么分开
- 元数据保存：抓取时间、URL、语言、来源类别、状态码要不要留
- 回溯能力：后面发现脏数据时，能不能追溯到原来源

所以这一节要建立的不是“会写一个爬虫”，而是“知道训练数据入口到底长什么样”。


## Different sources are different problems

不同来源不是同一类数据换个 URL 而已，它们连采集难点都不同。

### Web pages

网页的难点通常是：

- HTML 结构脏
- 模板和正文混在一起
- 广告、导航、评论区干扰很大
- 镜像页和模板页很多

### Code repositories

代码仓库的难点通常是：

- 不只是源代码，还有 README、issue、docstring、注释、提交信息
- 不同语言项目结构差异很大
- 自动生成文件、依赖锁文件、vendor 目录可能污染很重

### Papers and technical documents

论文和技术文档常见难点是：

- PDF 文本抽取质量参差不齐
- 多栏布局、公式、脚注、引用容易打乱正文顺序
- HTML 版和 PDF 版的结构不一致

### Forums and discussion data

论坛和问答数据的难点通常是：

- 楼中楼结构复杂
- 引用、回复、签名、模板文本很多
- 同一个问题常常有大量低质量灌水回复

也就是说，source type 不只是标签，而是决定后面抽取规则和清洗策略的重要信号。


In [ ]:
# 一个最小 source registry。
# 它的作用不是“记录网址列表”这么简单，而是先把来源当成被管理的对象。

source_registry = [
    {
        'source_name': 'official_docs',
        'source_type': 'web_docs',
        'priority': 'high',
        'crawl_budget': 2000,
        'refresh_policy': 'weekly',
    },
    {
        'source_name': 'code_repositories',
        'source_type': 'git_repo',
        'priority': 'high',
        'crawl_budget': 800,
        'refresh_policy': 'daily',
    },
    {
        'source_name': 'research_papers',
        'source_type': 'pdf_html',
        'priority': 'medium',
        'crawl_budget': 500,
        'refresh_policy': 'monthly',
    },
    {
        'source_name': 'forums',
        'source_type': 'discussion',
        'priority': 'medium',
        'crawl_budget': 300,
        'refresh_policy': 'weekly',
    },
]

for item in source_registry:
    print(item)


## What a crawl pipeline usually looks like

一个更像真实工程的数据采集管线，通常不是一条“下载文本”的直线，而更像下面这个过程：

1. 先从 source registry 或种子列表里取任务
2. 决定抓哪个 URL、repo、paper 或讨论串
3. 发送请求或读取内容
4. 检查 robots、频控、错误码、重试策略
5. 对网页做正文抽取，对代码仓库做文件筛选，对 PDF 做文本抽取
6. 保存原始内容和元数据
7. 按来源、语言、质量预判先分桶
8. 再把结果送往清洗和去重阶段

重点是：**采集阶段不是最后得到训练文本，而是先得到可追溯的原始记录。**


In [ ]:
crawl_pipeline = [
    'load_source_registry',
    'pick_next_target_from_frontier',
    'fetch_raw_content',
    'respect_robots_and_rate_limits',
    'extract_main_content_and_metadata',
    'store_raw_record',
    'bucket_by_source_and_language',
    'send_to_cleaning_stage',
]

for step_id, step in enumerate(crawl_pipeline, start=1):
    print(f'{step_id}. {step}')


## Why metadata matters

如果只存正文，不存元数据，后面很多工作都会变得很难：

- 发现异常时无法回溯来源
- 无法按语言、站点、抓取时间做分析
- 无法判断某条数据是不是旧版本
- 无法判断是否需要重新抓取

所以真正可用的原始记录，通常至少会保留：

- URL 或唯一源标识
- 来源类型
- 抓取时间
- 状态码
- 语言提示
- 正文文本
- 原始长度
- 哈希值
- 分桶信息


In [ ]:
# 一个更像真实工程输入记录的 toy 版本。
# 重点不是字段是否完整，而是理解“原始记录”和“训练文本”不是同一个层次的东西。

raw_record = {
    'url': 'https://example.com/article',
    'source_type': 'web_article',
    'source_name': 'official_docs',
    'fetch_time': '2026-05-26T22:00:00Z',
    'status_code': 200,
    'language_hint': 'en',
    'html_length': 128734,
    'main_text': 'This is the extracted body text...',
    'sha256': 'sha256:example',
    'raw_bucket': 'raw_web_high_priority',
}

for key, value in raw_record.items():
    print(f'{key}: {value}')


## Main-content extraction is a bottleneck, not a detail

对网页类数据来说，正文抽取经常是最关键的一层。因为 HTML 里混着太多“看起来也是文字，但不该进训练集”的东西：

- 顶栏导航
- 侧边栏推荐
- Cookie 提示
- 页脚法律文本
- 评论区碎片
- 广告位内容

如果这一层做不好，后面的去重、质量过滤和训练都会被污染。


In [ ]:
# 一个最小正文抽取示意。
# 真实系统会复杂得多，这里只强调“不是所有文字都该保留”。

page_segments = {
    'title': 'Example Article',
    'nav': 'Home Docs Blog About',
    'main': 'This is the extracted body text with actual content.',
    'comments': 'first! thanks! useful post!',
    'footer': 'Privacy Terms Contact',
}

# 这里我们故意只保留 title 和 main，模拟正文抽取器的选择。
kept_fields = ['title', 'main']
extracted_text = ' '.join(page_segments[field] for field in kept_fields)

dropped_fields = [field for field in page_segments if field not in kept_fields]

print('kept_fields =', kept_fields)
print('dropped_fields =', dropped_fields)
print('extracted_text =', extracted_text)


## Revisit policy, failures, and why frontier management exists

采集系统还有一个经常被忽略的问题：它不是抓一次就永远结束。

你需要回答：

- 某些文档站是不是要周期性重抓？
- 某些失败页面要不要重试？
- 某些来源是不是已经价值很低，不值得继续消耗预算？
- 某些 URL 是不是已经抓过，而且内容没有变化？

这就是 frontier management 和 revisit policy 的意义。它们决定资源怎么分配，也决定后面数据湖会不会被低价值抓取淹没。


In [ ]:
# 一个 toy frontier 调度例子。

frontier = [
    {'target': 'docs_page_1', 'priority': 10, 'last_fetch_days_ago': 10, 'status': 'stale'},
    {'target': 'repo_readme', 'priority': 8, 'last_fetch_days_ago': 1, 'status': 'fresh'},
    {'target': 'forum_thread_42', 'priority': 5, 'last_fetch_days_ago': 20, 'status': 'retry'},
]

# 简单排序：优先级高的先抓；同优先级下，越旧的越优先。
frontier = sorted(frontier, key=lambda x: (-x['priority'], -x['last_fetch_days_ago']))

for item in frontier:
    print(item)


## Common mistakes

- 以为抓到 HTML 就等于拿到了训练文本。真正可用的是正文加元数据。
- 把所有来源都当成同一种抓取对象，忽略网页、代码、论文、论坛之间的结构差异。
- 不做 source registry 和预算管理，结果资源大量浪费在低价值来源上。
- 不保留元数据，后面很难回溯问题来源和做按桶修复。
- 把正文抽取当小问题，实际它常常决定后面清洗成本上限。

## Summary

这一节最重要的不是某条爬虫规则，而是建立一个正确视角：训练数据入口本身就是工程系统。它必须同时考虑来源分层、抓取策略、正文抽取、元数据保存和回溯能力。只有这样，后面的清洗、去重和训练才有可靠原材料可用。
